# 06 — Caso completo: pipeline de exportação guiado por configuração

Tudo junto, num sistema pequeno mas realista: um pipeline que **lê** vendas de um CSV, **transforma** e **entrega** em vários destinos —
com tudo declarado em YAML.

```mermaid
flowchart LR
    subgraph config [configuração]
        B[pipeline.yaml] --> M{deep merge}
        P[pipeline.prod.yaml] --> M
        O[overrides: CLI / env] --> M
    end
    M -->|PipelineSpec<br/>Pydantic, uniões discriminadas, enums| S[spec validada e congelada]
    S -->|registro Spec → componente| R[Pipeline<br/>dataclasses com comportamento]
    R --> E[execução: source → transforms → sinks]
    CSV[vendas.csv] --> E
    E --> V[Venda<br/>Pydantic: linha a linha, com nº da linha]
```

| peça | ferramenta | notebook de origem |
|---|---|---|
| formatos e merge de camadas | YAML + `deep_merge` | [01](01_formatos_de_configuracao.ipynb), [03](03_configuracao_tipada.ipynb) |
| vocabulários (`kind`, ambiente, região) | `StrEnum` | [02](02_factory_pattern.ipynb) |
| validação da configuração | Pydantic, união discriminada, `Literal` dinâmico | [02](02_factory_pattern.ipynb), [04](04_politicas_e_presets.ipynb) |
| componentes de runtime | dataclasses `frozen` + `slots`, registro por decorador | [02](02_factory_pattern.ipynb) |
| dados de entrada | Pydantic com `BeforeValidator` | [03](03_configuracao_tipada.ipynb) |

In [1]:
import csv
import io
import json
import re
import tempfile
from collections.abc import Callable, Iterable, Iterator
from dataclasses import dataclass, field
from datetime import date
from decimal import Decimal
from enum import StrEnum
from pathlib import Path
from typing import Annotated, Any, Literal, Protocol

import yaml
from pydantic import (BaseModel, BeforeValidator, ConfigDict, Field, HttpUrl, SecretStr, TypeAdapter, ValidationError,
                      model_validator)

CONFIGS = Path("configs")

## 1. O dado: `Venda`

O CSV tem números no formato brasileiro (`1.234,50`) e uma região fora do vocabulário. O modelo da linha decide o que é válido:

In [2]:
class Regiao(StrEnum):
    NORTE = "norte"
    NORDESTE = "nordeste"
    CENTRO_OESTE = "centro-oeste"
    SUDESTE = "sudeste"
    SUL = "sul"


BRL_NUMBER = re.compile(r"\d{1,3}(\.\d{3})*(,\d{1,2})?|\d+(,\d{1,2})?")


def parse_brl(value: Any) -> Any:
    """'1.234,50' -> '1234.50'; recusa formatos ambíguos como '1.5' (ver caveats)."""
    if isinstance(value, str):
        text = value.strip()
        if not BRL_NUMBER.fullmatch(text):
            raise ValueError(f"valor fora do formato 1.234,56: {value!r}")
        return text.replace(".", "").replace(",", ".")
    return value


ValorBRL = Annotated[Decimal, Field(ge=0, decimal_places=2), BeforeValidator(parse_brl)]


class Venda(BaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True, str_strip_whitespace=True)
    data: date
    regiao: Regiao
    produto: str = Field(min_length=1)
    valor: ValorBRL


print(f"Venda(...): {Venda(data='2026-11-01', regiao='sul', produto='café', valor='1.234,50')!r}")

Venda(...): Venda(data=datetime.date(2026, 11, 1), regiao=<Regiao.SUL: 'sul'>, produto='café', valor=Decimal('1234.50'))


## 2. A especificação do pipeline

Cada parte com vocabulário fechado e união discriminada. Um detalhe que só dá para fazer com o modelo do dado em mãos: o campo do
filtro é um `Literal` **gerado a partir dos campos de `Venda`** — um filtro por `"regaio"` falha na carga da configuração, não
silenciosamente na execução.

In [3]:
class Environment(StrEnum):
    DEV = "dev"
    PROD = "prod"


class LogLevel(StrEnum):
    DEBUG = "debug"
    INFO = "info"
    WARNING = "warning"


VendaField = Literal[tuple(Venda.model_fields)]


class Spec(BaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True)


class CsvSourceSpec(Spec):
    kind: Literal["csv"]
    path: Path
    delimiter: str = Field(default=",", min_length=1, max_length=1)


class FilterSpec(Spec):
    kind: Literal["filter"]
    field: VendaField
    equals: str


class MinValueSpec(Spec):
    kind: Literal["min_value"]
    value: Decimal = Field(ge=0)


class FileSinkSpec(Spec):
    kind: Literal["file"]
    path: Path


class ConsoleSinkSpec(Spec):
    kind: Literal["console"]
    max_rows: int = Field(default=5, ge=1)


class WebhookSinkSpec(Spec):
    kind: Literal["webhook"]
    url: HttpUrl
    retries: int = Field(default=3, ge=0, le=10)
    token: SecretStr = SecretStr("")


SourceSpec = Annotated[CsvSourceSpec, Field(discriminator="kind")]
TransformSpec = Annotated[FilterSpec | MinValueSpec, Field(discriminator="kind")]
SinkSpec = Annotated[FileSinkSpec | ConsoleSinkSpec | WebhookSinkSpec, Field(discriminator="kind")]


class PipelineSpec(Spec):
    name: str = Field(pattern=r"^[a-z0-9-]+$")
    environment: Environment
    log_level: LogLevel = LogLevel.INFO
    source: SourceSpec
    transforms: tuple[TransformSpec, ...] = ()
    sinks: tuple[SinkSpec, ...] = Field(min_length=1)

    @model_validator(mode="after")
    def prod_rules(self):
        if self.environment is Environment.PROD:
            if any(isinstance(s, ConsoleSinkSpec) for s in self.sinks):
                raise ValueError("em prod, sink 'console' não é permitido")
            if self.log_level is LogLevel.DEBUG:
                raise ValueError("em prod, log_level debug não é permitido")
        return self


class ConfigFile(Spec):
    pipeline: PipelineSpec

## 3. Carregando as camadas

`deep_merge` mescla dicionários recursivamente. A decisão que precisa ser **explícita** é o que acontece com listas: aqui a lista
da camada de cima **substitui** a de baixo (concatenar faria a produção herdar o sink `console` da base).

In [4]:
def deep_merge(base: dict, override: dict, path: str = "") -> dict:
    merged = dict(base)
    for key, value in override.items():
        where = f"{path}.{key}" if path else str(key)
        if value is None and isinstance(merged.get(key), dict):
            raise ValueError(f"{where}: valor vazio apagaria a seção inteira da camada de baixo (ver caveats)")
        if isinstance(value, dict) and isinstance(merged.get(key), dict):
            merged[key] = deep_merge(merged[key], value, where)
        else:
            merged[key] = value                          # escalares e LISTAS: substituem
    return merged


class ConfigError(Exception):
    pass


def load_pipeline(environment: str = "dev", overrides: dict | None = None) -> PipelineSpec:
    layers = [CONFIGS / "pipeline.yaml"]
    if environment != "dev":
        layers.append(CONFIGS / f"pipeline.{environment}.yaml")
    data: dict = {}
    try:
        for layer in layers:
            data = deep_merge(data, yaml.safe_load(layer.read_text()) or {})
        data = deep_merge(data, overrides or {})
    except ValueError as ex:
        raise ConfigError(f"camadas inconsistentes: {ex}") from None
    try:
        return ConfigFile.model_validate(data).pipeline
    except ValidationError as ex:
        names = " + ".join(layer.name for layer in layers) + (" + overrides" if overrides else "")
        raise ConfigError(f"configuração inválida ({names}):\n  " + "\n  ".join(readable_errors(ex))) from None


def readable_errors(ex: ValidationError) -> list[str]:
    """Esconde 'too_short' de uma coleção quando o motivo real é um item inválido dentro dela."""
    errors = ex.errors()
    item_locs = [e["loc"] for e in errors]
    lines = []
    for e in errors:
        is_child_of_this = any(len(other) > len(e["loc"]) and other[: len(e["loc"])] == e["loc"] for other in item_locs)
        if e["type"] == "too_short" and is_child_of_this:
            continue
        lines.append(f"{'.'.join(map(str, e['loc']))}: {e['msg'].removeprefix('Value error, ')}")
    return lines


dev = load_pipeline("dev")
prod = load_pipeline("prod", overrides={"pipeline": {"sinks": [{"kind": "file", "path": "saida/vendas.jsonl"},
                                                               {"kind": "webhook", "url": "https://hooks.example.com/vendas",
                                                                "retries": 5, "token": "s3cr3t"}]}})
print(f"dev.sinks: {[s.kind for s in dev.sinks]}")
print(f"prod.sinks: {[s.kind for s in prod.sinks]}")
print(f"prod.log_level: {prod.log_level!r}")
print(f"prod.transforms (herdado da base): {[t.kind for t in prod.transforms]}")

dev.sinks: ['file', 'console']
prod.sinks: ['file', 'webhook']
prod.log_level: <LogLevel.WARNING: 'warning'>
prod.transforms (herdado da base): ['filter', 'min_value']


Configuração efetiva para auditoria — o segredo sai mascarado:

In [5]:
print(f"prod.model_dump_json(indent=2, exclude_none=True):\n{prod.model_dump_json(indent=2, exclude_none=True)}")

prod.model_dump_json(indent=2, exclude_none=True):
{
  "name": "vendas-diarias",
  "environment": "prod",
  "log_level": "warning",
  "source": {
    "kind": "csv",
    "path": "vendas.csv",
    "delimiter": ";"
  },
  "transforms": [
    {
      "kind": "filter",
      "field": "regiao",
      "equals": "sul"
    },
    {
      "kind": "min_value",
      "value": "50"
    }
  ],
  "sinks": [
    {
      "kind": "file",
      "path": "saida/vendas.jsonl"
    },
    {
      "kind": "webhook",
      "url": "https://hooks.example.com/vendas",
      "retries": 5,
      "token": "**********"
    }
  ]
}


## 4. Componentes de runtime e o registro Spec → componente

Os componentes são dataclasses **stdlib** (`frozen`, `slots`): não sabem nada de YAML nem de Pydantic. O registro liga cada tipo de
spec à função que constrói o componente.

In [6]:
class Source(Protocol):
    def read(self, base: Path) -> Iterator[tuple[int, dict[str, str]]]: ...


class Transform(Protocol):
    def apply(self, rows: Iterable[Venda]) -> Iterator[Venda]: ...


class Sink(Protocol):
    def write(self, rows: list[Venda], base: Path) -> str: ...


BUILDERS: dict[type[Spec], Callable[[Any], Any]] = {}


def builds(spec_type: type[Spec]):
    def decorator(fn):
        if spec_type in BUILDERS:
            raise TypeError(f"{spec_type.__name__} já tem construtor")
        BUILDERS[spec_type] = fn
        return fn
    return decorator


@dataclass(frozen=True, slots=True)
class CsvSource:
    path: Path
    delimiter: str

    def read(self, base: Path) -> Iterator[tuple[int, dict[str, str]]]:
        with open(base / self.path, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f, delimiter=self.delimiter)
            for row in reader:
                yield reader.line_num, row


@dataclass(frozen=True, slots=True)
class FilterEquals:
    field: str
    equals: str

    def apply(self, rows: Iterable[Venda]) -> Iterator[Venda]:
        return (r for r in rows if str(getattr(r, self.field)) == self.equals)


@dataclass(frozen=True, slots=True)
class MinValue:
    value: Decimal

    def apply(self, rows: Iterable[Venda]) -> Iterator[Venda]:
        return (r for r in rows if r.valor >= self.value)


@dataclass(frozen=True, slots=True)
class JsonlFile:
    path: Path

    def write(self, rows: list[Venda], base: Path) -> str:
        target = base / self.path
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text("".join(row.model_dump_json() + "\n" for row in rows), encoding="utf-8")
        return f"file: {len(rows)} linhas em {self.path}"


@dataclass(frozen=True, slots=True)
class Console:
    max_rows: int

    def write(self, rows: list[Venda], base: Path) -> str:
        shown = "; ".join(f"{r.data} {r.regiao} {r.produto} {r.valor}" for r in rows[: self.max_rows])
        return f"console: {shown}" + (f" (+{len(rows) - self.max_rows})" if len(rows) > self.max_rows else "")


@dataclass(frozen=True, slots=True)
class Webhook:
    url: str
    retries: int
    has_token: bool

    def write(self, rows: list[Venda], base: Path) -> str:              # simulado: sem rede
        payload = json.dumps([row.model_dump(mode="json") for row in rows])
        return f"webhook: POST {self.url} ({len(payload)} bytes, até {self.retries} tentativas, token={'sim' if self.has_token else 'não'})"


builds(CsvSourceSpec)(lambda s: CsvSource(s.path, s.delimiter))
builds(FilterSpec)(lambda s: FilterEquals(s.field, s.equals))
builds(MinValueSpec)(lambda s: MinValue(s.value))
builds(FileSinkSpec)(lambda s: JsonlFile(s.path))
builds(ConsoleSinkSpec)(lambda s: Console(s.max_rows))
builds(WebhookSinkSpec)(lambda s: Webhook(str(s.url), s.retries, bool(s.token.get_secret_value())))


def union_members(annotation: Any) -> set[type]:
    from typing import get_args
    inner = get_args(annotation)[0]                     # Annotated[X | Y, ...] -> X | Y
    return set(get_args(inner)) or {inner}


every_spec = union_members(SourceSpec) | union_members(TransformSpec) | union_members(SinkSpec)
print(f"specs sem construtor: {sorted(t.__name__ for t in every_spec - set(BUILDERS))}")

specs sem construtor: []


Esse último teste é o mais importante do padrão de registro: adicionar um tipo novo de sink à união e esquecer o construtor é
pego **antes** de qualquer execução.

## 5. Montagem e execução

In [7]:
@dataclass(frozen=True, slots=True)
class Pipeline:
    name: str
    source: Source
    transforms: tuple[Transform, ...]
    sinks: tuple[Sink, ...]

    def run(self, base: Path) -> dict[str, Any]:
        valid, rejected = [], []
        for line, raw in self.source.read(base):
            try:
                valid.append(Venda.model_validate(raw))
            except ValidationError as ex:
                rejected.append((line, [f"{e['loc'][0]}: {e['msg']}" for e in ex.errors()]))
        rows: Iterable[Venda] = valid
        for transform in self.transforms:
            rows = transform.apply(rows)
        selected = list(rows)
        return {"lidas": len(valid) + len(rejected), "rejeitadas": rejected, "selecionadas": len(selected),
                "entregas": [sink.write(selected, base) for sink in self.sinks]}


def build_pipeline(spec: PipelineSpec) -> Pipeline:
    make = lambda s: BUILDERS[type(s)](s)
    return Pipeline(spec.name, make(spec.source), tuple(map(make, spec.transforms)), tuple(map(make, spec.sinks)))


with tempfile.TemporaryDirectory() as tmp:
    workdir = Path(tmp)
    (workdir / "vendas.csv").write_bytes((CONFIGS / "vendas.csv").read_bytes())
    for label, spec in [("dev", dev), ("prod", prod)]:
        report = build_pipeline(spec).run(workdir)
        print(f"[{label}] lidas={report['lidas']} selecionadas={report['selecionadas']}")
        for line, errors in report["rejeitadas"]:
            print(f"[{label}]   linha {line} rejeitada: {errors}")
        for delivery in report["entregas"]:
            print(f"[{label}]   {delivery}")
    written = (workdir / "saida" / "vendas.jsonl").read_text().splitlines()
    print(f"conteúdo do arquivo (1ª linha): {written[0]}")

[dev] lidas=7 selecionadas=2
[dev]   linha 5 rejeitada: ["regiao: Input should be 'norte', 'nordeste', 'centro-oeste', 'sudeste' or 'sul'"]
[dev]   linha 6 rejeitada: ["valor: Value error, valor fora do formato 1.234,56: 'abc'"]
[dev]   file: 2 linhas em saida/vendas.jsonl
[dev]   console: 2026-11-01 sul café 1234.50; 2026-11-04 sul chá 2000.00
[prod] lidas=7 selecionadas=2
[prod]   linha 5 rejeitada: ["regiao: Input should be 'norte', 'nordeste', 'centro-oeste', 'sudeste' or 'sul'"]
[prod]   linha 6 rejeitada: ["valor: Value error, valor fora do formato 1.234,56: 'abc'"]
[prod]   file: 2 linhas em saida/vendas.jsonl
[prod]   webhook: POST https://hooks.example.com/vendas (169 bytes, até 5 tentativas, token=sim)
conteúdo do arquivo (1ª linha): {"data":"2026-11-01","regiao":"sul","produto":"café","valor":"1234.50"}


Leitura do resultado: 7 linhas lidas; a linha 5 (região `oeste`) e a linha 6 (valor `abc`) foram rejeitadas **com número de
linha**; dos válidos, o filtro deixou só o `sul` com valor ≥ 50.

## 6. Configurações inválidas: o que quem edita o YAML vê

(O `readable_errors` acima existe por um motivo visto na prática: quando o único sink é inválido, o Pydantic descarta o item e
reporta **também** `sinks: Tuple should have at least 1 item` — verdadeiro, mas ruído para quem editou o arquivo.)

In [8]:
broken_overrides = {
    "campo do filtro com erro de digitação": {"pipeline": {"transforms": [{"kind": "filter", "field": "regaio", "equals": "sul"}]}},
    "sink desconhecido": {"pipeline": {"sinks": [{"kind": "s3", "bucket": "x"}]}},
    "webhook sem https e retries demais": {"pipeline": {"sinks": [{"kind": "webhook", "url": "ftp//x", "retries": 50}]}},
    "chave com erro de digitação": {"pipeline": {"source": {"kind": "csv", "path": "vendas.csv", "delimitter": ";"}}},
    "nome fora do padrão": {"pipeline": {"name": "Vendas Diárias"}},
}
for label, overrides in broken_overrides.items():
    try:
        load_pipeline("dev", overrides)
    except ConfigError as ex:
        print(f"# {label}\n{ex}\n")

# campo do filtro com erro de digitação
configuração inválida (pipeline.yaml + overrides):
  pipeline.transforms.0.filter.field: Input should be 'data', 'regiao', 'produto' or 'valor'

# sink desconhecido
configuração inválida (pipeline.yaml + overrides):
  pipeline.sinks.0: Input tag 's3' found using 'kind' does not match any of the expected tags: 'file', 'console', 'webhook'

# webhook sem https e retries demais
configuração inválida (pipeline.yaml + overrides):
  pipeline.sinks.0.webhook.url: Input should be a valid URL, relative URL without a base
  pipeline.sinks.0.webhook.retries: Input should be less than or equal to 10

# chave com erro de digitação
configuração inválida (pipeline.yaml + overrides):
  pipeline.source.csv.delimitter: Extra inputs are not permitted

# nome fora do padrão
configuração inválida (pipeline.yaml + overrides):
  pipeline.name: String should match pattern '^[a-z0-9-]+$'



E as regras de produção, que só existem na camada de cima:

In [9]:
for label, overrides in {
    "console em prod": {"pipeline": {"sinks": [{"kind": "console"}]}},
    "debug em prod": {"pipeline": {"log_level": "debug"}},
}.items():
    try:
        load_pipeline("prod", overrides)
    except ConfigError as ex:
        print(f"# {label}\n{ex}\n")

# console em prod
configuração inválida (pipeline.yaml + pipeline.prod.yaml + overrides):
  pipeline: em prod, sink 'console' não é permitido

# debug em prod
configuração inválida (pipeline.yaml + pipeline.prod.yaml + overrides):
  pipeline: em prod, log_level debug não é permitido



## ⚠️ Pegadinhas, efeitos colaterais e caveats

### Já demonstradas acima

| # | pegadinha | onde | como evitar |
|---|---|---|---|
| 1 | concatenar listas no merge faria a produção herdar o `console` da base | §3 | regra explícita: listas substituem |
| 2 | um item inválido numa coleção gera também `too_short` na coleção | §6 | filtrar o erro "pai" (`readable_errors`) |
| 3 | campo de filtro com erro de digitação só falharia na execução | §2 | `Literal` gerado dos campos de `Venda` |
| 4 | segredo em log/auditoria | §3 | `SecretStr` (mas veja o notebook 03: dump não serve para recarregar) |
| 5 | tipo de sink novo sem construtor | §4 | teste de cobertura do registro |

### Novas (as duas primeiras eram **bugs deste notebook**, já corrigidos acima)

**1. Número em formato brasileiro: a conversão ingênua erra por 10× sem avisar.**

In [10]:
def parse_brl_naive(value: str) -> str:
    return value.strip().replace(".", "").replace(",", ".")


for raw in ["1.234,50", "1,5", "1.5", "1.234"]:
    try:
        strict = parse_brl(raw)
    except ValueError as ex:
        strict = f"erro ({ex})"
    print(f"{raw!r}: ingênuo={parse_brl_naive(raw)!r}  estrito={strict}")

'1.234,50': ingênuo='1234.50'  estrito=1234.50
'1,5': ingênuo='1.5'  estrito=1.5
'1.5': ingênuo='15'  estrito=erro (valor fora do formato 1.234,56: '1.5')
'1.234': ingênuo='1234'  estrito=1234


`"1.5"` (escrito em formato americano numa planilha) virava `15` — dez vezes o valor, aceito pela validação de `Decimal`. A versão
atual exige o formato completo (`1.234,56`) e recusa o ambíguo. `"1.234"` continua aceito como mil duzentos e trinta e quatro:
dentro do formato brasileiro, é isso que significa.

**2. Seção vazia numa camada de cima apaga a seção inteira de baixo.** Em YAML, `pipeline:` sem conteúdo é `None` — e um merge
ingênuo troca o dicionário inteiro por `None`:

In [11]:
def deep_merge_naive(base: dict, override: dict) -> dict:
    merged = dict(base)
    for key, value in override.items():
        merged[key] = deep_merge_naive(merged[key], value) if isinstance(value, dict) and isinstance(merged.get(key), dict) else value
    return merged


base_layer = yaml.safe_load((CONFIGS / "pipeline.yaml").read_text())
empty_override = yaml.safe_load("pipeline:\n")             # alguém comentou todas as linhas da seção
print(f"merge ingênuo: {deep_merge_naive(base_layer, empty_override)}")
try:
    load_pipeline("dev", overrides=empty_override)
except ConfigError as ex:
    print(f"merge atual: {ex}")

merge ingênuo: {'pipeline': None}
merge atual: camadas inconsistentes: pipeline: valor vazio apagaria a seção inteira da camada de baixo (ver caveats)


O erro de validação que viria depois ("`pipeline` obrigatório") não diz **qual camada** apagou o quê. Escalares vazios
(`retries:`) continuam passando como `None` explícito — aí a validação do campo responde.

**3. Gerador consumido duas vezes: o segundo destino recebe nada.** As transformações devolvem geradores (preguiçosos). Se dois
sinks iterassem o mesmo gerador:

In [12]:
rows = [Venda(data="2026-11-01", regiao="sul", produto="café", valor="100,00"),
        Venda(data="2026-11-02", regiao="sul", produto="chá", valor="200,00")]
lazy_rows = FilterEquals("regiao", "sul").apply(rows)
print(f"primeiro sink: {len(list(lazy_rows))} linhas")
print(f"segundo sink: {len(list(lazy_rows))} linhas")

primeiro sink: 2 linhas
segundo sink: 0 linhas


Sem erro: o segundo destino recebe um arquivo vazio. Por isso `Pipeline.run` materializa (`selected = list(rows)`) antes de
entregar. Gerador é ótimo para **um** consumidor; para vários, materialize ou use `itertools.tee`.

**4. CSV irregular: colunas sobrando viram a chave `None`; faltando, valor `None`.**

In [13]:
ragged = io.StringIO("data;regiao;produto;valor\n2026-11-01;sul;café;10,00;EXTRA\n2026-11-02;sul;chá\n")
for line_rows in csv.DictReader(ragged, delimiter=";"):
    try:
        Venda.model_validate(line_rows)
    except ValidationError as ex:
        print(f"{line_rows} -> {[(e['loc'], e['type']) for e in ex.errors()]}")

{'data': '2026-11-01', 'regiao': 'sul', 'produto': 'café', 'valor': '10,00', None: ['EXTRA']} -> [(('None',), 'invalid_key')]
{'data': '2026-11-02', 'regiao': 'sul', 'produto': 'chá', 'valor': None} -> [(('valor',), 'decimal_type')]


O Pydantic rejeita, mas a mensagem (`('None',)`, `invalid_key`) não ajuda ninguém. Melhor checar a forma da linha antes (número de
colunas, `None in row`) e dizer "linha 2: 5 colunas, esperadas 4". Sem `extra="forbid"` no modelo, a coluna extra seria
**descartada em silêncio**.

**5. Efeitos parciais: o primeiro destino escreve, o segundo falha.**

In [14]:
@dataclass(frozen=True, slots=True)
class BrokenSink:
    def write(self, rows: list[Venda], base: Path) -> str:
        raise ConnectionError("webhook fora do ar")


with tempfile.TemporaryDirectory() as tmp:
    workdir = Path(tmp)
    (workdir / "vendas.csv").write_bytes((CONFIGS / "vendas.csv").read_bytes())
    partial = Pipeline("parcial", CsvSource(Path("vendas.csv"), ";"), (), (JsonlFile(Path("saida/vendas.jsonl")), BrokenSink()))
    try:
        partial.run(workdir)
    except ConnectionError as ex:
        print(f"falhou: {ex}")
    print(f"mas o arquivo ficou: {(workdir / 'saida' / 'vendas.jsonl').exists()}")

falhou: webhook fora do ar
mas o arquivo ficou: True


Não existe transação entre destinos diferentes. Opções, da mais simples à mais robusta: (1) ordenar os destinos do mais propenso a
falhar para o menos; (2) capturar o erro **por destino** e reportar o que foi e o que não foi entregue; (3) escrever em arquivo
temporário e renomear no fim (`os.replace` é atômico no mesmo sistema de arquivos); (4) destinos **idempotentes**, que podem ser
reexecutados com segurança.

**6. Reexecução sobrescreve a saída.** `JsonlFile` grava com `write_text`: rodar duas vezes no mesmo dia troca o arquivo, sem
histórico. Isso é idempotência (bom) se o conteúdo é sempre o mesmo, e perda de dados (ruim) se a entrada mudou entre as execuções.
A decisão — sobrescrever, versionar pelo nome (`vendas-2026-11-11.jsonl`) ou acrescentar — é parte da especificação do sink,
não detalhe de implementação.

## 7. Verificações finais

In [15]:
assert every_spec <= set(BUILDERS)
assert [s.kind for s in dev.sinks] == ["file", "console"]
assert [s.kind for s in prod.sinks] == ["file", "webhook"]
assert "s3cr3t" not in prod.model_dump_json()
with tempfile.TemporaryDirectory() as tmp:
    (Path(tmp) / "vendas.csv").write_bytes((CONFIGS / "vendas.csv").read_bytes())
    report = build_pipeline(dev).run(Path(tmp))
    assert report["lidas"] == 7 and [line for line, _ in report["rejeitadas"]] == [5, 6]
    assert report["selecionadas"] == 2
print("verificações: ok")

verificações: ok


## O que este caso mostra

- **A configuração é validada inteira antes de qualquer efeito colateral** (nenhum arquivo é lido ou escrito com configuração inválida).
- **Três tipos de classe, três papéis:** `StrEnum` para vocabulários, Pydantic na fronteira (configuração e dados de entrada),
  dataclasses stdlib para o comportamento.
- **A fábrica é declarativa** (uniões discriminadas) na validação e **um dicionário** na construção — com teste de cobertura.
- **As camadas têm regra explícita** para listas, e regras de produção vivem no modelo, não em `if` espalhados.